# 基于图文嵌入特征相似度进行性别分类

In [ ]:
# 运行准备：复用p5lib加载CLIP和学生数据，并按示例代码5.71～5.72提取前置嵌入特征
import numpy as np
from PIL import Image
import sentence_transformers as sentrans
from p5lib.ch5 import load_sentence_model, load_student_portrait_data

model = load_sentence_model("./fm/clip-ViT-B-32")
label_txt = ['a female portrait', 'a male portrait']
label_emb = model.encode(label_txt)
students, img_path_list = load_student_portrait_data()
img_emb = [None]*len(students)
for i, img_path in enumerate(img_path_list):
    img = Image.open(img_path_list[i])
    img = img.resize((224, 224))
    img_emb[i] = model.encode(img)
img_emb = np.array(img_emb)

In [ ]:
scores = sentrans.util.cos_sim(img_emb, label_emb)
pred = np.argmax(scores, axis=1)
pred = pred.cpu().numpy()
new_table = students[['姓名', '性别']].copy()
new_table['女性相似度'] = [f"{v:.2f}" for v in scores[:, 0]]
new_table['男性相似度'] = [f"{v:.2f}" for v in scores[:, 1]]
new_table['预测标签'] = pred
new_table.T